# Data Preparation for Linear Probing Analysis

## Overview
This notebook prepares multilingual datasets for probing the internal representations of the Tiny Aya model across 13 diverse languages. We create two complementary probing tasks:

1. **Language Identification Probing**: Using FLORES+ parallel corpus to test if the model can distinguish between languages
2. **Part-of-Speech Tagging Probing**: Using Universal Dependencies treebanks to test syntactic understanding

## Target Languages
- **High-resource**: English, Spanish, French, German  
- **Medium-resource**: Arabic, Hindi, Bengali, Tamil, Turkish, Persian
- **Low-resource**: Swahili, Amharic, Yoruba

## Datasets
- **FLORES+**: 500 samples per language for language identification
- **Universal Dependencies**: 100 sentences per language for POS tagging

The prepared data will be used to train linear probes on different layers of Tiny Aya to understand how linguistic knowledge emerges across the model's depth.



In [5]:
_DATASET_DIR = "../datasets"

In [6]:
# Load HF_TOKEN from .env file
import os
from dotenv import load_dotenv

load_dotenv()

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    print("HF_TOKEN successfully loaded from .env file.")
else:
    print("HF_TOKEN not found in .env file. Proceeding without authentication.")

HF_TOKEN successfully loaded from .env file.


In [10]:
# Language family and script mapping
language_metadata = {
    'English': {'script': 'Latin', 'family': 'Indo-European', 'resource_level': 'High'},
    'Spanish': {'script': 'Latin', 'family': 'Indo-European', 'resource_level': 'High'},
    'French': {'script': 'Latin', 'family': 'Indo-European', 'resource_level': 'High'},
    'German': {'script': 'Latin', 'family': 'Indo-European', 'resource_level': 'High'},
    'Arabic': {'script': 'Arabic', 'family': 'Afro-Asiatic', 'resource_level': 'Medium'},
    'Hindi': {'script': 'Devanagari', 'family': 'Indo-European', 'resource_level': 'Medium'},
    'Bengali': {'script': 'Bengali', 'family': 'Indo-European', 'resource_level': 'Medium'},
    'Tamil': {'script': 'Tamil', 'family': 'Dravidian', 'resource_level': 'Medium'},
    'Turkish': {'script': 'Latin', 'family': 'Turkic', 'resource_level': 'Medium'},
    'Persian': {'script': 'Arabic', 'family': 'Indo-European', 'resource_level': 'Medium'},
    'Swahili': {'script': 'Latin', 'family': 'Niger-Congo', 'resource_level': 'Low'},
    'Amharic': {'script': 'Ethiopic', 'family': 'Afro-Asiatic', 'resource_level': 'Low'},
    'Yoruba': {'script': 'Latin', 'family': 'Niger-Congo', 'resource_level': 'Low'}
}

# Create short language code for label
lang_code_mapping = {
    'English': 'en', 'Spanish': 'es', 'French': 'fr', 'German': 'de',
    'Arabic': 'ar', 'Hindi': 'hi', 'Bengali': 'bn', 'Tamil': 'ta',
    'Turkish': 'tr', 'Persian': 'fa', 'Swahili': 'sw', 'Amharic': 'am', 'Yoruba': 'yo'
}

In [13]:
import json
from datasets import load_dataset
from collections import Counter
from tqdm import tqdm


# Configuration: Mapping your 13 languages to FLORES+ configs
# format: { 'human_readable_name': 'flores_plus_config_name' }
languages = {
    'English': 'eng_Latn',
    'Spanish': 'spa_Latn',
    'French': 'fra_Latn',
    'German': 'deu_Latn',
    'Arabic': 'arb_Arab',
    'Hindi': 'hin_Deva',
    'Bengali': 'ben_Beng',
    'Tamil': 'tam_Taml',
    'Turkish': 'tur_Latn',
    'Persian': 'pes_Arab',
    'Swahili': 'swh_Latn',
    'Amharic': 'amh_Ethi',
    'Yoruba': 'yor_Latn'
}

SAMPLES_PER_LANG = 500
output_data = []


for display_name, config in languages.items():
    print(f"Loading dataset for {display_name} ({config})...")
    
    try:
        dataset = load_dataset("openlanguagedata/flores_plus", config, split='dev', trust_remote_code=True)
        
        # Shuffle to ensure randomness for selected subset
        dataset = dataset.shuffle(seed=42)
        
        # Take exactly 500 samples
        subset = dataset.select(range(min(SAMPLES_PER_LANG, len(dataset))))
        
        for item in tqdm(subset):
            output_data.append({
                "id": f"{lang_code_mapping[display_name]}_{item['id']:03d}",
                "text": item['text'],
                "language": display_name.lower(),
                "label": lang_code_mapping[display_name],
                "task": "language_id",
                "metadata": language_metadata.get(display_name, {'script': 'Unknown', 'family': 'Unknown', 'resource_level': 'Unknown'})
            })
            
    except Exception as e:
        print(f"❌ Error loading {display_name}: {e}")



print(f"\n Done! Prepared {len(output_data)} examples.")
print(f"Stats: {dict(Counter(d['label'] for d in output_data))}")

Loading dataset for English (eng_Latn)...


100%|██████████| 500/500 [00:00<00:00, 15216.49it/s]


Loading dataset for Spanish (spa_Latn)...


100%|██████████| 500/500 [00:00<00:00, 19283.45it/s]


Loading dataset for French (fra_Latn)...


...

Loading dataset for Amharic (amh_Ethi)...


100%|██████████| 500/500 [00:00<00:00, 16110.38it/s]


Loading dataset for Yoruba (yor_Latn)...


100%|██████████| 500/500 [00:00<00:00, 11943.12it/s]


 Done! Prepared 6500 examples.
Stats: {'en': 500, 'es': 500, 'fr': 500, 'de': 500, 'ar': 500, 'hi': 500, 'bn': 500, 'ta': 500, 'tr': 500, 'fa': 500, 'sw': 500, 'am': 500, 'yo': 500}


In [14]:
output_data[0]

{'id': 'en_972',
 'text': 'Nevertheless, all French-speaking Belgians and Swiss would have learned standard French in school, so they would be able to understand you even if you used the standard French numbering system.',
 'language': 'english',
 'label': 'en',
 'task': 'language_id',
 'metadata': {'script': 'Latin',
  'family': 'Indo-European',
  'resource_level': 'High'}}

In [15]:
# Save to JSON
import os
os.makedirs(_DATASET_DIR, exist_ok=True)

with open(os.path.join(_DATASET_DIR, 'language_id_probing_data.json'), 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)